# Projectile Motion from Wave Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/notebooks/LFM_Projectile_Motion.ipynb)

## What This Notebook Demonstrates

A wave packet in a $\chi$-gradient follows a **parabolic trajectory** — the same as projectile motion under gravity.

- No `F = mg` is injected
- The $\chi$-gradient bends the wave path (GOV-01 only)
- We fit the trajectory and check for parabolic shape ($R^2 > 0.90$)

**Equation**: $\partial^2 E/\partial t^2 = c^2 \nabla^2 E - \chi^2 E$ with $\chi(y) = \chi_0 + g \cdot y$

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi_background = 1.0
c = 1.0
N = 150
L = 60.0
dx = L / N
dt = 0.3 * dx / c
n_steps = 800
gravity_strength = 0.02

x = np.linspace(0, L, N)
y = np.linspace(0, L, N)
X, Y = np.meshgrid(x, y, indexing='ij')
chi = chi_background + gravity_strength * Y

# Wave packet with horizontal velocity
proj_x0, proj_y0 = 10.0, 40.0
proj_width = 2.5
velocity_x = 0.5

r2 = (X - proj_x0)**2 + (Y - proj_y0)**2
E = np.exp(-r2 / (2 * proj_width**2))
r2_prev = (X - proj_x0 + velocity_x*dt)**2 + (Y - proj_y0)**2
E_prev = np.exp(-r2_prev / (2 * proj_width**2))

trajectory = []
for step in range(n_steps):
    laplacian = np.zeros_like(E)
    laplacian[1:-1, 1:-1] = (
        E[:-2, 1:-1] + E[2:, 1:-1] +
        E[1:-1, :-2] + E[1:-1, 2:] -
        4 * E[1:-1, 1:-1]
    ) / dx**2
    E_new = 2 * E - E_prev + dt**2 * (c**2 * laplacian - chi**2 * E)
    E_new[0, :] = E_new[-1, :] = E_new[:, 0] = E_new[:, -1] = 0
    E_prev = E.copy()
    E = E_new.copy()
    E2 = E**2
    total = np.sum(E2)
    if total > 1e-10:
        trajectory.append((step * dt, np.sum(X * E2) / total, np.sum(Y * E2) / total))

trajectory = np.array(trajectory)
t_c = trajectory[:, 0] - trajectory[0, 0]
x_pos = trajectory[:, 1]
y_pos = trajectory[:, 2]

# Parabolic fit to vertical
y_coeffs = np.polyfit(t_c, y_pos, 2)
y_fit = np.polyval(y_coeffs, t_c)
ss_res = np.sum((y_pos - y_fit)**2)
ss_tot = np.sum((y_pos - np.mean(y_pos))**2)
R2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
curves_down = y_coeffs[0] < 0

print('=' * 60)
print('PROJECTILE MOTION RESULTS')
print('=' * 60)
print(f'Effective acceleration: {2 * y_coeffs[0]:.6f}')
print(f'R² (parabolic fit):    {R2:.4f}')
print(f'Curves downward:       {curves_down}')
print(f'H0 REJECTED: {R2 > 0.90 and curves_down}')
if R2 > 0.90 and curves_down:
    print('CONCLUSION: Parabolic projectile motion EMERGES from GOV-01')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(x_pos, y_pos, 'b-', lw=2, label='Wave packet trajectory')
ax.plot(x_pos[0], y_pos[0], 'go', ms=10, label='Launch')
ax.plot(x_pos[-1], y_pos[-1], 'rs', ms=10, label='Landing')
ax.set_xlabel('Horizontal position')
ax.set_ylabel('Vertical position')
ax.set_title('Projectile Trajectory (wave packet in chi-gradient)')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(t_c, y_pos, 'b.', ms=2, label='Data')
ax.plot(t_c, y_fit, 'r-', lw=2, label=f'Parabolic fit (R²={R2:.3f})')
ax.set_xlabel('Time')
ax.set_ylabel('Vertical position')
ax.set_title('Vertical Motion: Parabolic Fit')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('LFM Projectile Motion from GOV-01', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Key Result

- A wave packet in a $\chi$-gradient curves downward in a **parabolic arc**
- This IS projectile motion — emerging purely from GOV-01 wave dynamics
- No force law was injected; the $\chi$-gradient bends the wave path